# YOLO 워크숍 시작 노트북 — 우리 팀 프로젝트

이 노트북은 워크숍 프로젝트의 **뼈대**입니다. 주제와 데이터만 정하면, 데이터 받기 → 형식 맞추기 → 학습 → 평가 → 개선 실험 → 결과 정리까지 같은 순서로 진행할 수 있습니다.
`✏️` 표시가 있는 셀은 우리 팀 내용으로 **직접 채우는** 곳입니다.

| 단계 | 할 일 | 체크포인트 |
|---|---|---|
| 1 | 프로젝트 카드 작성 | CP1 주제·데이터 확정 |
| 2 | 데이터 받기 (캐글 / URL / Drive / 예시 데이터) | CP1 |
| 3 | 형식 확인·변환·분할 → `data.yaml`(데이터 위치·클래스 이름을 적은 설정 파일) | CP1 |
| 4 | 데이터 살펴보기 | CP2 |
| 5 | 베이스라인(개선 전 기준점이 되는 첫 모델) 학습 | CP2 첫 학습 |
| 6 | 평가 — 지표, 임계값(threshold, 예측을 남길지 버릴지 가르는 기준값), 오류 분석 | CP3 |
| 7 | 개선 실험 기록 | CP3 개선 |
| 8 | 데모 — 새 이미지·영상에 적용 | CP4 |
| 9 | 결과 정리 — 요약 리포트 자동 생성 | CP4 결과 공유 |

**실행 환경**: Google Colab, 런타임(Colab이 빌려주는 실행 환경) 유형 **T4 GPU**(Colab 무료 GPU 종류). 결과는 Google Drive `MyDrive/yolo_workshop/<프로젝트 이름>`에 저장되므로 다음 시간에 이어서 할 수 있습니다.

---
## 1. 프로젝트 카드 ✏️

아래 셀의 값을 우리 팀 내용으로 바꾸고 실행합니다. 이 내용은 마지막 결과 리포트에 그대로 들어갑니다.
- `PROJECT_NAME`은 **영문·숫자·밑줄**만 씁니다 (폴더 이름으로 쓰임).
- `SUCCESS`에는 '무엇을 얼마나 잘하면 성공인지'를 숫자로 적습니다. 예: `안전모 미착용을 테스트 이미지에서 재현율(recall — 실제 정답 중 찾아낸 비율) 0.8 이상으로 찾는다`

In [ ]:
PROJECT_NAME = 'ppe_helmet_check'          # ✏️ 영문·숫자·밑줄 (Drive 결과 폴더 이름이 됨)
TEAM = '홍길동, 김철수'                     # ✏️ 리포트의 Team 항목에 들어감
PROBLEM = '공사 현장 사진에서 안전모를 쓰지 않은 작업자를 찾는다'          # ✏️ 누가 어떤 상황에서 무엇을 알고 싶은가
SUCCESS = '테스트 세트에서 no_helmet 클래스 재현율 0.8 이상'            # ✏️ 숫자로 (마지막에 실제 결과와 비교)
TASK = 'detect'                             # 'detect'(박스) 또는 'segment'(마스크) — 기본 모델이 바뀜

---
## 준비 — 설치와 저장 위치

In [ ]:
# ultralytics(YOLO), kagglehub(캐글 다운로드), tabulate(마크다운 표)
!pip -q install ultralytics kagglehub tabulate

In [ ]:
# 표준 라이브러리: 파일·경로·압축·XML 처리
import os, re, json, glob, time, shutil, random, zipfile
from pathlib import Path
from collections import Counter, defaultdict
import xml.etree.ElementTree as ET

# 이미지(cv2)·수치(numpy)·표(pandas)·그래프·딥러닝(torch)
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import Image, display
from ultralytics import YOLO

print('GPU 사용 가능:', torch.cuda.is_available())  # False면 런타임 유형을 T4 GPU로 바꿀 것

# Colab에서 실행 중인지 확인 (Colab 전용 모듈을 불러와 봄)
try:
    from google.colab import drive, files, userdata  # Drive 연결, 파일 올리기, 보안 비밀 읽기
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# fullmatch: 이름 전체가 영문·숫자·밑줄인지 검사
# 검사 → Drive 연결 → 결과 저장 폴더(SAVE_DIR) 만들기
assert re.fullmatch(r'[A-Za-z0-9_]+', PROJECT_NAME), 'PROJECT_NAME은 영문·숫자·밑줄만 쓰세요'
if IN_COLAB:
    drive.mount('/content/drive')  # 처음 한 번 권한 허용 창이 뜸
    SAVE_DIR = Path('/content/drive/MyDrive/yolo_workshop') / PROJECT_NAME  # Path끼리 / 로 이으면 하위 경로
else:
    SAVE_DIR = Path('outputs') / PROJECT_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)  # 중간 폴더까지 생성, 이미 있으면 그대로
RAW = Path('raw_data')           # 받은 원본 (런타임 로컬 디스크)
DATA = Path('dataset')           # YOLO 형식으로 정리한 데이터
print('결과 저장 위치:', SAVE_DIR.resolve())

# save_out: 만든 파일을 Drive 결과 폴더에 복사
# 입력: 파일 경로 / 출력: Drive 쪽 경로
def save_out(path):
    dst = SAVE_DIR / Path(path).name  # 같은 파일 이름으로 Drive 폴더에
    if Path(path).resolve() != dst.resolve():  # 이미 Drive 안의 파일이면 복사 생략
        shutil.copy(path, dst)
    return dst

# show_grid: 이미지 여러 장을 격자로 표시
# 입력: BGR 이미지 목록, 제목 목록, 열 수(cols), 칸 크기(size)
def show_grid(imgs, titles=None, cols=3, size=4):
    rows = (len(imgs) + cols - 1) // cols  # 올림 나눗셈: 필요한 행 수
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows))
    for ax, i in zip(np.array(axes).ravel(), range(rows * cols)):  # 2차원 축 배열을 1줄로 펴서 차례로
        ax.axis('off')  # 눈금 숨김 (남는 칸은 빈칸)
        if i < len(imgs):
            ax.imshow(imgs[i][..., ::-1])  # [..., ::-1] 채널 뒤집기: OpenCV BGR → RGB
            if titles:
                ax.set_title(titles[i], fontsize=9)
    plt.tight_layout(); plt.show()

---
## 2. 데이터 받기 ✏️

`DATA_SOURCE`를 하나 고르고, 그 아래 해당 값만 채웁니다.

| `DATA_SOURCE` | 언제 | 채울 값 |
|---|---|---|
| `'kaggle'` | 캐글(Kaggle, 공개 데이터셋 공유 사이트) 공개 데이터셋 | `KAGGLE_SLUG` — 데이터셋 주소 `kaggle.com/datasets/` 뒤의 `작성자/이름` |
| `'url'` | 공공데이터·GitHub 등에서 **zip 직접 다운로드 링크**가 있을 때 | `ZIP_URL` |
| `'drive'` | AI Hub처럼 따로 받아 **내 Drive에 올려 둔 zip** | `DRIVE_ZIP` |
| `'ultralytics'` | 연습용 예시 데이터 (설정 이름만으로 자동 다운로드) | `ULTRA_YAML` (예: `'construction-ppe.yaml'`) |

**캐글 인증** — 공개 데이터셋은 로그인 없이 받아지는 경우가 많습니다. 다운로드 오류(401/403 — 인증·권한 문제를 뜻하는 HTTP 오류 번호)가 나면 인증을 설정합니다.
1. 캐글 웹사이트 → 오른쪽 위 프로필 → **Settings → API**에서 토큰(token, 내 계정임을 증명하는 비밀 문자열)을 만듭니다.
2. Colab 왼쪽의 **열쇠 아이콘(보안 비밀 — Secrets, 토큰 같은 비밀 값을 코드 밖에 저장하는 Colab 기능)** 에 `KAGGLE_API_TOKEN`이라는 이름으로 토큰 값을 등록하고 '노트북 액세스'를 켭니다.
   예전 방식의 `kaggle.json`(username, key)을 받았다면 `KAGGLE_USERNAME`, `KAGGLE_KEY` 두 개로 등록합니다.
3. 코드에 키를 직접 적지 마세요. 노트북을 공유하면 키도 함께 퍼집니다.

In [ ]:
# 고른 방법에 해당하는 값만 쓰이고 나머지는 무시됨
DATA_SOURCE = 'ultralytics'                       # ✏️ 'kaggle' | 'url' | 'drive' | 'ultralytics' — 받는 방법
KAGGLE_SLUG = 'snehilsanyal/construction-site-safety-image-dataset-roboflow'   # ✏️ 'kaggle'일 때: 주소의 작성자/이름
ZIP_URL = ''                                      # ✏️ 'url'일 때: zip 직접 다운로드 링크
DRIVE_ZIP = '/content/drive/MyDrive/my_data.zip'  # ✏️ 'drive'일 때: Drive 안 zip 경로
ULTRA_YAML = 'construction-ppe.yaml'              # ✏️ 'ultralytics'일 때: 예시 데이터 설정 이름

In [ ]:
# 입력: 폴더 경로 / zip을 같은 이름 폴더에 풀고 zip은 삭제
def unzip_all(src_dir):
    """폴더 안의 zip을 모두 풀기 (압축 안에 압축이 있는 경우 대비)"""
    for z in list(Path(src_dir).rglob('*.zip')):  # rglob: 하위 폴더까지 모두 검색
        with zipfile.ZipFile(z) as f:
            f.extractall(z.parent / z.stem)  # a.zip → 같은 위치의 a/ 폴더에
        z.unlink()  # 풀고 난 zip 삭제 (디스크 절약)

# 원본 폴더 준비. ULTRA_DATA는 예시 데이터일 때만 채워짐
RAW.mkdir(exist_ok=True)
ULTRA_DATA = None
if DATA_SOURCE == 'kaggle':
    if IN_COLAB:                       # 보안 비밀에 등록된 값이 있으면 사용
        for key in ['KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY']:
            try:
                os.environ[key] = userdata.get(key)  # 환경 변수로 넘기면 kagglehub가 인증에 사용
            except Exception:  # 등록 안 된 이름은 건너뜀
                pass
    import kagglehub
    path = kagglehub.dataset_download(KAGGLE_SLUG)  # 캐시 폴더에 받고 그 경로를 돌려줌
    shutil.copytree(path, RAW / 'kaggle', dirs_exist_ok=True)  # raw_data/kaggle로 복사
elif DATA_SOURCE == 'url':
    assert ZIP_URL, 'ZIP_URL을 채우세요'
    # wget: 링크의 파일을 raw_data/download.zip으로 저장 (-q 조용히)
    !wget -q -O raw_data/download.zip "{ZIP_URL}"
elif DATA_SOURCE == 'drive':
    shutil.copy(DRIVE_ZIP, RAW / Path(DRIVE_ZIP).name)  # Drive의 zip을 런타임 디스크로 복사
elif DATA_SOURCE == 'ultralytics':
    from ultralytics.data.utils import check_det_dataset
    ULTRA_DATA = check_det_dataset(ULTRA_YAML)  # 없으면 자동 다운로드, 설정 dict 반환
    shutil.copytree(ULTRA_DATA['path'], RAW / 'ultralytics', dirs_exist_ok=True)  # 'path' = 데이터가 받아진 폴더
else:
    raise ValueError(DATA_SOURCE)  # 지원하지 않는 값(오타 등)이면 멈춤
unzip_all(RAW)  # 받은 것 중 zip이 있으면 모두 풀기

# 받은 파일을 확장자별로 세어 이미지·라벨 종류 확인
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}  # 이미지로 인정할 확장자
ext_count = Counter(p.suffix.lower() for p in RAW.rglob('*') if p.is_file())  # 예: {'.jpg': 3000, '.txt': 3000}
print('받은 파일 종류:', dict(ext_count.most_common(8)))

In [ ]:
# 폴더 구조 훑어보기 (깊이 3까지, 폴더마다 파일 수)
def tree(root, depth=3):
    root = Path(root)
    for d in sorted([root] + [p for p in root.rglob('*') if p.is_dir()]):  # 자기 자신 + 모든 하위 폴더
        level = len(d.relative_to(root).parts)  # root 기준 깊이 (root = 0)
        if level > depth:  # 너무 깊은 폴더는 생략
            continue
        n = sum(1 for f in d.iterdir() if f.is_file())  # 이 폴더 바로 안의 파일 수
        print('    ' * level + f'{d.name}/  ({n} files)')  # 깊이만큼 들여쓰기
tree(RAW)

---
## 3. 형식 확인과 변환 → `data.yaml`

공개 데이터셋은 라벨 형식이 제각각입니다. 아래 셀이 **자동으로 형식을 알아내** YOLO 형식으로 바꿉니다.

| 형식 | 생김새 | 처리 |
|---|---|---|
| YOLO | 이미지마다 같은 이름의 `.txt`, 한 줄 = `클래스 cx cy w h` (0~1, 박스 중심·크기를 이미지 크기로 나눈 정규화 값) | 그대로 사용 |
| Pascal VOC(PASCAL VOC 대회에서 쓰던 XML 라벨 형식) | 이미지마다 같은 이름의 `.xml`, 픽셀 좌표 `xmin ymin xmax ymax` | YOLO로 변환 |
| COCO(MS COCO 데이터셋의 JSON 라벨 형식) | `.json` 하나에 모든 이미지와 박스, `bbox = [x, y, w, h]` 픽셀 | YOLO로 변환 |

원본에 `train`(학습용)/`val`(`valid`, 학습 중 성능 확인용)/`test`(마지막 평가용) 폴더가 있으면 그 구분을 따르고, 없으면 7:2:1로 무작위 분할합니다(`SEED` = 난수 시작값, 고정해 두면 다시 실행해도 같은 분할).
YOLO 형식인데 클래스 이름 파일(`data.yaml`, `classes.txt`, `obj.names`)이 없으면 `CLASS_NAMES`를 직접 채워야 합니다.

In [ ]:
CLASS_NAMES = None        # ✏️ YOLO 형식인데 클래스 이름 파일이 없을 때만: ['helmet', 'head', ...] (순서 = 클래스 번호)
SPLIT_RATIO = (0.7, 0.2, 0.1)  # train:val:test 비율 (원본에 분할이 없을 때만 사용)
SEED = 0  # 난수 시작값 — 같으면 매번 같은 분할

# detect_format: 라벨 파일 종류를 보고 데이터 형식을 추정
# 입력: 원본 폴더 / 출력: (형식 이름, 라벨 파일 목록)
# 형식 이름: 'voc' | 'coco' | 'yolo' | 'unknown'
# 판단 순서: xml 10개 초과 → VOC, COCO 구조 json → COCO, txt 10개 초과 → YOLO
def detect_format(root):
    xmls = list(root.rglob('*.xml'))  # Pascal VOC 후보: 모든 xml
    jsons = [j for j in root.rglob('*.json') if j.stat().st_size > 200]  # 200바이트 이하 작은 json(설정 파일 등)은 제외
    coco = []  # COCO 구조가 확인된 json만 모음
    for j in jsons:
        try:
            d = json.load(open(j))
            if isinstance(d, dict) and 'annotations' in d and 'images' in d:  # COCO 필수 키 확인
                coco.append(j)
        except Exception:  # json이 아니거나 깨진 파일은 무시
            pass
    # YOLO 후보: txt 중 클래스 이름·설명 파일은 제외
    txts = [t for t in root.rglob('*.txt') if t.name not in ('classes.txt', 'README.txt', 'readme.txt')]
    if len(xmls) > 10:  # 10개 이하면 우연히 섞인 파일로 보고 무시
        return 'voc', xmls
    if coco:
        return 'coco', coco
    if len(txts) > 10:
        return 'yolo', txts
    return 'unknown', []  # 지원 형식 없음 → 폴더 구조 출력을 보고 확인

# 입력: 파일 경로 / 출력: 'train' | 'val' | 'test' | None
# 예: raw_data/kaggle/valid/labels/a.txt → 'val'
def split_of(path):
    """경로에 들어 있는 train/val/test 폴더 이름 → 분할 이름 (없으면 None)"""
    parts = [p.lower() for p in Path(path).parts]  # 경로를 폴더 이름 목록으로 (소문자)
    # (분할 이름, 그 분할로 인정할 폴더 이름들) 순서대로 검사
    for name, keys in [('train', ('train', 'training')), ('val', ('val', 'valid', 'validation')), ('test', ('test', 'testing'))]:
        if any(k in parts for k in keys):  # 폴더 이름 전체가 같아야 인정 (부분 일치 X)
            return name
    return None

# 형식 판정 실행 + 이미지를 '이름 → 경로' 사전으로 정리
FMT, LABEL_FILES = detect_format(RAW)
images = {p.stem: p for p in RAW.rglob('*') if p.suffix.lower() in IMG_EXT}  # stem = 확장자 뺀 이름, 같은 이름이면 마지막 것만 남음
print('라벨 형식:', FMT, '| 라벨 파일', len(LABEL_FILES), '개 | 이미지', len(images), '장')

In [ ]:
# voc_records: Pascal VOC xml → YOLO 박스 목록
# 입력: xml 경로 목록 / 출력: (레코드 목록, 클래스 이름 목록)
# 레코드 = (이미지 stem, [(클래스 번호, cx, cy, w, h), ...], 분할 이름)
# cx·cy·w·h는 박스 중심·크기를 이미지 크기로 나눈 0~1 값 (YOLO 형식)
def voc_records(xmls):
    recs, names = [], []  # names: 처음 나온 순서대로 클래스 번호 부여
    for x in xmls:
        r = ET.parse(x).getroot()  # xml 최상위 <annotation> 요소
        fn = r.findtext('filename') or x.stem  # <filename> 값, 없으면 xml 파일 이름
        stem = Path(fn).stem if Path(fn).stem in images else x.stem  # 그 이름의 이미지가 실제로 있으면 사용
        size = r.find('size')  # <size> 안에 <width>, <height>
        w, h = float(size.findtext('width') or 0), float(size.findtext('height') or 0)  # 이미지 너비·높이 (픽셀), 없으면 0
        if (w == 0 or h == 0) and stem in images:  # 크기 정보가 없으면 이미지를 직접 읽음
            h, w = cv2.imread(str(images[stem])).shape[:2]  # shape = (높이, 너비, 채널)
        boxes = []
        for o in r.findall('object'):  # 물체(박스) 하나당 <object> 하나
            n = o.findtext('name').strip()  # 클래스 이름
            if n not in names:  # 새 클래스면 목록 끝에 추가
                names.append(n)
            bb = o.find('bndbox')
            x1, y1, x2, y2 = (float(bb.findtext(k)) for k in ('xmin', 'ymin', 'xmax', 'ymax'))  # 픽셀 좌표: 왼쪽 위, 오른쪽 아래
            # 모서리 → 중심(cx, cy)·너비·높이, 이미지 크기로 나눠 0~1로
            boxes.append((names.index(n), (x1 + x2) / 2 / w, (y1 + y2) / 2 / h, (x2 - x1) / w, (y2 - y1) / h))
        recs.append((stem, boxes, split_of(x)))  # 분할은 xml이 있던 폴더 이름으로 판단
    return recs, names

# coco_records: COCO json → YOLO 박스 목록 (출력 형식은 voc_records와 같음)
# COCO bbox = [x, y, w, h] 픽셀, x·y는 박스 왼쪽 위 모서리
def coco_records(jsons):
    recs, names, cat_index = [], [], {}  # cat_index: (json, COCO 클래스 id) → 우리 번호
    for j in jsons:  # json이 여러 개면(train/valid/test 등) 차례로
        d = json.load(open(j))
        for c in d['categories']:  # 클래스 목록
            if c['name'] not in names:
                names.append(c['name'])
            cat_index[(j, c['id'])] = names.index(c['name'])  # COCO id는 0부터가 아닐 수 있어 다시 매김
        by_img = defaultdict(list)  # 이미지 id → 그 이미지의 박스 목록
        for a in d['annotations']:  # 박스 하나 = annotation 하나
            by_img[a['image_id']].append(a)
        for im in d['images']:
            w, h = im['width'], im['height']  # 이미지 크기 (픽셀)
            # 중심 = 왼쪽 위 + 크기/2, 모두 이미지 크기로 나눠 0~1로
            boxes = [(cat_index[(j, a['category_id'])], (a['bbox'][0] + a['bbox'][2] / 2) / w,
                      (a['bbox'][1] + a['bbox'][3] / 2) / h, a['bbox'][2] / w, a['bbox'][3] / h)
                     for a in by_img[im['id']]]
            recs.append((Path(im['file_name']).stem, boxes, split_of(j)))  # 분할은 json이 있던 폴더 이름으로
    return recs, names

# yolo_records: YOLO txt를 읽어 레코드로 정리 (출력 형식은 위와 같음)
# 다각형(세그멘테이션) 라벨은 감싸는 박스로 바꿈, 클래스 이름은 파일에서 찾음
def yolo_records(txts):
    recs = []
    for t in txts:
        if t.stem not in images:  # 짝이 되는 이미지가 없는 txt는 건너뜀
            continue
        boxes = []
        for line in t.read_text().strip().splitlines():  # 한 줄 = 물체 하나
            v = line.split()  # 공백으로 나눈 값 목록 (문자열)
            if len(v) == 5:  # 클래스 cx cy w h → 박스 그대로
                boxes.append((int(v[0]), *map(float, v[1:])))  # *: 실수 4개를 풀어 튜플에 넣음
            elif len(v) > 5 and len(v) % 2 == 1:          # 클래스 x1 y1 x2 y2 … (다각형) → 감싸는 박스
                xs, ys = list(map(float, v[1::2])), list(map(float, v[2::2]))  # [1::2] = x들, [2::2] = y들 (한 칸 건너뛰기)
                # x·y의 최소~최대 범위 = 감싸는 박스의 중심·너비·높이
                boxes.append((int(v[0]), (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)))
        recs.append((t.stem, boxes, split_of(t)))
    names = CLASS_NAMES  # 직접 입력한 이름이 있으면 우선 사용
    if names is None:  # 없으면 원본 폴더에서 이름 파일 찾기
        # 후보: *.yaml(names 항목), classes.txt, *.names(한 줄에 이름 하나)
        for cand in list(RAW.rglob('*.yaml')) + list(RAW.rglob('classes.txt')) + list(RAW.rglob('*.names')):
            if cand.suffix == '.yaml':
                import yaml
                y = yaml.safe_load(open(cand))  # yaml → 파이썬 dict
                if isinstance(y, dict) and 'names' in y:
                    n = y['names']
                    names = [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)  # {0: 'a', 1: 'b'}든 리스트든 번호 순서로
                    break
            else:
                names = [l.strip() for l in cand.read_text().splitlines() if l.strip()]  # 빈 줄을 뺀 각 줄 = 클래스 이름
                break
    assert names, '클래스 이름을 찾지 못했습니다. 위 셀의 CLASS_NAMES를 채우세요'
    return recs, names

# 예시 데이터는 설정에 든 클래스 이름을 그대로 사용
if ULTRA_DATA is not None and FMT == 'yolo' and CLASS_NAMES is None:
    CLASS_NAMES = [ULTRA_DATA['names'][k] for k in sorted(ULTRA_DATA['names'])]
# 형식에 맞는 함수를 골라 실행 (FMT가 'unknown'이면 KeyError)
RECS, NAMES = {'voc': voc_records, 'coco': coco_records, 'yolo': yolo_records}[FMT](LABEL_FILES)
print('라벨 묶음', len(RECS), '개 | 클래스', len(NAMES), '개:', NAMES)

In [ ]:
# build_dataset: 레코드를 YOLO 폴더 구조로 저장하고 data.yaml 작성
# 입력: 레코드 목록, 클래스 이름 / 출력: (data.yaml 경로, 분할별 장 수, 원본 분할 유지 여부)
# 결과: dataset/images/{train,val,test}/, dataset/labels/{train,val,test}/
def build_dataset(recs, names):
    if DATA.exists():
        shutil.rmtree(DATA)  # 이전에 만든 dataset 폴더는 지우고 새로
    rng = random.Random(SEED)  # SEED로 고정한 난수 생성기 (분할 재현용)
    has_split = sum(r[2] is not None for r in recs) > 0.9 * len(recs)  # 90% 넘게 분할 폴더 안에 있으면 원본 분할 사용
    counts = Counter()  # 분할별 이미지 수
    for stem, boxes, sp in recs:
        if stem not in images:  # 이미지 파일이 없는 라벨은 건너뜀
            continue
        if not has_split or sp is None:  # 분할 정보가 없으면 무작위로 정함
            u = rng.random()  # 0~1 난수
            # u < 0.7 → train, 0.7~0.9 → val, 나머지 → test (기본 7:2:1)
            sp = 'train' if u < SPLIT_RATIO[0] else 'val' if u < SPLIT_RATIO[0] + SPLIT_RATIO[1] else 'test'
        src = images[stem]  # 원본 이미지 경로
        # 분할 폴더를 만들고 이미지 복사
        (DATA / 'images' / sp).mkdir(parents=True, exist_ok=True)
        (DATA / 'labels' / sp).mkdir(parents=True, exist_ok=True)
        shutil.copy(src, DATA / 'images' / sp / src.name)
        # YOLO 한 줄 = '클래스 cx cy w h', min·max로 0~1 밖의 값을 잘라 냄
        lines = [f'{c} {min(max(cx, 0), 1):.6f} {min(max(cy, 0), 1):.6f} {min(w, 1):.6f} {min(h, 1):.6f}'
                 for c, cx, cy, w, h in boxes if w > 0 and h > 0]  # 너비·높이가 0 이하인 박스는 버림
        # 이미지와 같은 이름의 txt로 저장 (박스가 없으면 빈 파일 = 배경 이미지)
        (DATA / 'labels' / sp / (src.stem + '.txt')).write_text('\n'.join(lines) + ('\n' if lines else ''))
        counts[sp] += 1
    if counts['val'] == 0:                     # 원본에 val이 없으면 train의 1/5을 val로
        tr = sorted((DATA / 'images' / 'train').iterdir())  # train 이미지 목록
        rng.shuffle(tr)  # 섞은 뒤 앞쪽을 val로
        (DATA / 'images' / 'val').mkdir(parents=True, exist_ok=True); (DATA / 'labels' / 'val').mkdir(parents=True, exist_ok=True)
        for p in tr[:max(1, len(tr) // 5)]:  # 최소 1장, train의 20%
            # 이미지와 라벨을 함께 옮겨야 짝이 맞음
            shutil.move(p, DATA / 'images' / 'val' / p.name)
            shutil.move(DATA / 'labels' / 'train' / (p.stem + '.txt'), DATA / 'labels' / 'val' / (p.stem + '.txt'))
            counts['train'] -= 1; counts['val'] += 1
    # data.yaml 내용: 데이터 위치, 분할 폴더, 클래스 번호 → 이름
    cfg = {'path': str(DATA.resolve()), 'train': 'images/train', 'val': 'images/val',
           'names': {i: n for i, n in enumerate(names)}}
    if counts['test']:  # test 분할이 있을 때만 등록
        cfg['test'] = 'images/test'
    (DATA / 'data.yaml').write_text(json.dumps(cfg, ensure_ascii=False, indent=1))  # JSON 문법으로 써도 YAML로 읽힘
    return str(DATA / 'data.yaml'), dict(counts), has_split

# 실행: 레코드 → dataset/ 폴더와 data.yaml
DATA_YAML, SPLIT_COUNTS, KEPT_SPLIT = build_dataset(RECS, NAMES)
print('원본 분할 유지:' if KEPT_SPLIT else '무작위 7:2:1 분할:', SPLIT_COUNTS)
print('데이터 설정 파일:', DATA_YAML)
print(open(DATA_YAML).read())

---
## 4. 데이터 살펴보기

학습 전에 데이터를 눈으로 확인합니다(EDA — 탐색적 데이터 분석, 그래프는 `eda.png`로 저장). 여기서 발견한 문제(클래스 불균형(클래스마다 박스 수 차이가 큰 상태), 아주 작은 물체, 잘못된 라벨)가 뒤의 성능을 대부분 설명합니다.

In [ ]:
# 모든 라벨 파일을 읽어 '박스 하나 = 한 행'인 표(DataFrame) 만들기
rows = []
for sp in ['train', 'val', 'test']:
    # 분할 폴더가 없으면 빈 목록으로 건너뜀
    for f in (DATA / 'labels' / sp).glob('*.txt') if (DATA / 'labels' / sp).exists() else []:
        for line in f.read_text().strip().splitlines():
            c, cx, cy, w, h = line.split()  # 값은 문자열이라 아래에서 숫자로 바꿈
            rows.append({'split': sp, 'class': NAMES[int(c)], 'w': float(w), 'h': float(h)})
boxes_df = pd.DataFrame(rows)
boxes_df['area_pct'] = boxes_df['w'] * boxes_df['h'] * 100  # 박스 넓이가 이미지의 몇 %인지 (w·h가 0~1)

# 왼쪽: train 클래스별 박스 수(value_counts), 오른쪽: 박스 크기 분포
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
boxes_df[boxes_df['split'] == 'train']['class'].value_counts().plot.bar(ax=axes[0], color='tab:blue')
axes[0].set_title('Boxes per class (train)'); axes[0].set_ylabel('boxes')
axes[1].hist(boxes_df['area_pct'], bins=40, color='tab:gray', log=True)  # log=True: 세로축 로그 (적은 칸도 보이게)
axes[1].set_xlabel('box area (% of image)'); axes[1].set_title('Box size distribution')
plt.tight_layout(); plt.savefig('eda.png', dpi=110); save_out('eda.png'); plt.show()  # 그림 저장 + Drive 복사
print('이미지 1% 미만 크기의 작은 박스 비율:', round((boxes_df['area_pct'] < 1).mean(), 3))  # 높으면 imgsz 키우기 실험 후보

In [ ]:
# 정답 라벨을 그린 샘플 6장
# 클래스마다 고정된 무작위 색 (클래스 번호를 난수 씨앗으로)
palette = [tuple(int(c) for c in np.random.default_rng(i).integers(60, 255, 3)) for i in range(len(NAMES))]
samples = sorted((DATA / 'images' / 'train').iterdir())  # train 이미지 목록
random.Random(SEED).shuffle(samples)  # 고정 순서로 섞어 매번 같은 6장
imgs, titles = [], []
for p in samples[:6]:
    im = cv2.imread(str(p)); h, w = im.shape[:2]  # BGR 이미지, 높이·너비
    # 같은 이름의 라벨 txt 읽기
    lbl = (DATA / 'labels' / 'train' / (p.stem + '.txt')).read_text().strip().splitlines()
    for line in lbl:
        c, cx, cy, bw, bh = line.split(); c = int(c)
        # YOLO(중심·크기, 0~1) → 픽셀 모서리: (중심 ± 크기/2) × 이미지 크기
        x1, y1 = int((float(cx) - float(bw) / 2) * w), int((float(cy) - float(bh) / 2) * h)
        x2, y2 = int((float(cx) + float(bw) / 2) * w), int((float(cy) + float(bh) / 2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), palette[c], 2)  # 두께 2 사각형
        # 박스 위에 클래스 이름 (화면 위로 넘치지 않게 y ≥ 12)
        cv2.putText(im, NAMES[c], (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, palette[c], 2)
    imgs.append(im); titles.append(f'{p.name[:25]} | {len(lbl)} boxes')
show_grid(imgs, titles)

**확인해 보기** ✏️ — 아래 질문의 답을 이 셀 아래에 메모해 두세요. 멘토링 때 함께 봅니다.
- 클래스마다 박스 수가 몇 배나 차이 나나요? 적은 클래스는 성능이 낮게 나올 가능성이 큽니다.
- 박스가 아주 작은 이미지가 많나요? 그렇다면 `imgsz`(학습·예측 때 이미지를 맞추는 한 변 크기, 픽셀)를 키우는 실험이 필요합니다.
- 샘플 그림에서 빠진 라벨이나 이상한 박스가 보이나요?

---
## 5. 베이스라인 학습

먼저 **작은 모델, 기본 설정**으로 끝까지 한 번 돌려 기준점(베이스라인)을 만듭니다. 개선은 그다음입니다.
`QUICK_RUN = True`로 두면 5 epoch(에폭 — 학습 데이터 전체를 한 번 다 보는 단위)만 돌려 파이프라인(데이터 준비부터 평가까지 이어지는 처리 흐름)이 끝까지 도는지 빠르게 확인할 수 있습니다.
데이터가 커서 1 epoch가 10분 넘게 걸리면 `FRACTION`(fraction, 학습에 쓸 이미지 비율)을 0.3처럼 낮춰 학습 이미지의 일부만 씁니다.

In [ ]:
QUICK_RUN = True                              # ✏️ True: 5 epoch 빠른 점검 / False: 50 epoch 본 학습
BASE_MODEL = 'yolo26n.pt' if TASK == 'detect' else 'yolo26n-seg.pt'  # n = 가장 작은 모델, -seg = 마스크까지 내는 모델
FRACTION = 1.0                                # ✏️ 학습에 쓸 이미지 비율 (0.3 등으로 낮추면 빨라지지만 성능↓)
# 학습 설정: epochs 반복 횟수, imgsz 입력 크기(px), batch 한 번에 넣는 장 수
CFG = dict(epochs=5, imgsz=640, batch=16, fraction=FRACTION) if QUICK_RUN else dict(epochs=50, imgsz=640, batch=16, fraction=FRACTION)

# train: 모델 하나를 학습하고 best.pt를 Drive에 복사
# 입력: 실험 이름, 시작 모델, 바꿀 설정 / 출력: (best.pt 경로, 학습 분, 사용한 설정)
def train(name, model=BASE_MODEL, **kw):
    cfg = {**CFG, **kw}  # 기본 설정에 kw로 받은 값을 덮어씀
    m = YOLO(model)  # 사전학습 가중치 불러오기 (없으면 자동 다운로드)
    t0 = time.time()  # 학습 시간 측정 시작
    # 결과는 runs_ws/<name>/에 저장, exist_ok=True면 같은 폴더 재사용
    m.train(data=DATA_YAML, project='runs_ws', name=name, exist_ok=True, seed=SEED, verbose=False, **cfg)
    best = Path(m.trainer.save_dir) / 'weights' / 'best.pt'  # 검증 성능이 가장 좋았던 epoch의 가중치
    shutil.copy(best, SAVE_DIR / f'{name}_best.pt')  # Drive에 <이름>_best.pt로 보관
    return best, round((time.time() - t0) / 60, 1), cfg

BEST, BASE_MIN, BASE_CFG = train('baseline')  # 베이스라인 학습 실행
print(f'학습 {BASE_MIN}분 | 최고 가중치: {BEST} (Drive에도 저장)')
# results.png: epoch별 손실(loss)과 mAP 변화 그래프
display(Image(filename=str(Path(BEST).parent.parent / 'results.png'), width=900))

---
## 6. 평가 — 지표, 임계값, 오류 분석

### 6-1. 클래스별 지표

전체 mAP(평균 정밀도 — 여러 신뢰도 기준에서 정밀도를 평균한 탐지 성능 점수) 하나만 보면 안 됩니다. **우리 문제에서 중요한 클래스**의 재현율·정밀도(precision — 찾았다고 한 것 중 맞은 비율)를 따로 봐야 성공 기준(`SUCCESS`)과 비교할 수 있습니다.

In [ ]:
EVAL_SPLIT = 'test' if SPLIT_COUNTS.get('test') else 'val'  # test 분할이 있으면 test, 없으면 val로 평가
model = YOLO(BEST)  # 학습된 최고 가중치 불러오기
# 평가 실행: plots=True면 혼동 행렬·PR 곡선 그림도 저장
res = model.val(data=DATA_YAML, split=EVAL_SPLIT, verbose=False, plots=True, project='runs_ws', name='eval', exist_ok=True)
# 클래스별 정밀도(box.p)·재현율(box.r)·AP50·AP50-95 표
per_class = pd.DataFrame({
    'class': [NAMES[i] for i in res.box.ap_class_index],  # 평가 세트에 등장한 클래스 번호들
    'precision': np.round(res.box.p, 3), 'recall': np.round(res.box.r, 3),
    'ap50': np.round(res.box.ap50, 3), 'ap50_95': np.round(res.box.ap, 3)})  # box.ap = IoU 0.5~0.95 평균 AP
per_class.to_csv('per_class.csv', index=False); save_out('per_class.csv')  # CSV 저장 + Drive 복사
print(f'{EVAL_SPLIT} 세트 | mAP50 {res.box.map50:.3f} | mAP50-95 {res.box.map:.3f}')  # box.map50 = mAP50, box.map = mAP50-95
per_class

In [ ]:
EVAL_DIR = Path(res.save_dir)  # 평가 그림이 저장된 폴더
for name in ['confusion_matrix_normalized.png', 'BoxPR_curve.png']:  # 정규화 혼동 행렬, PR 곡선
    if (EVAL_DIR / name).exists():
        display(Image(filename=str(EVAL_DIR / name), width=520))

### 6-2. 신뢰도 임계값 정하기

서비스에서는 신뢰도(conf — 모델이 예측마다 매긴 0~1 확신 점수) 임계값 하나를 골라야 합니다. 평가 세트 예측을 한 번만 해 두고, 임계값을 바꿔 가며 **클래스를 구분해** TP(맞게 찾음)·FP(헛탐지)·FN(놓침)을 셉니다. (실습 A Step 2에서 만든 방법과 같습니다.)

In [ ]:
# iou_mat: 박스 묶음 a(N개)와 b(M개)의 IoU를 N×M 표로 계산
# 박스 형식 [x1, y1, x2, y2] 픽셀, IoU = 겹친 넓이 ÷ 합친 넓이
def iou_mat(a, b):
    if len(a) == 0 or len(b) == 0:  # 한쪽이 비면 빈 표
        return np.zeros((len(a), len(b)))
    # [:, None]·[None, :]로 차원을 늘려 모든 쌍을 한 번에 계산
    # 겹친 영역: 왼쪽 위는 둘 중 큰 값, 오른쪽 아래는 작은 값
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)  # np.clip(…, 0, None): 안 겹치면 음수 → 0
    # 각 박스의 넓이
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)  # 1e-9: 0으로 나누기 방지

# gt_boxes: 이미지 하나의 정답 라벨 → [x1, y1, x2, y2, 클래스] 픽셀 배열
def gt_boxes(img_path, split):
    h, w = cv2.imread(str(img_path)).shape[:2]  # 픽셀 변환에 쓸 이미지 크기
    out = []
    for line in (DATA / 'labels' / split / (img_path.stem + '.txt')).read_text().strip().splitlines():
        c, cx, cy, bw, bh = map(float, line.split())
        # 중심·크기(0~1) → 모서리 픽셀 좌표
        out.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h, c])
    return np.array(out).reshape(-1, 5)  # 박스가 없어도 (0, 5) 모양 유지

# 평가 이미지를 한 번만 예측해 cache에 저장 (임계값 바꿀 때 재사용)
EVAL_IMAGES = sorted((DATA / 'images' / EVAL_SPLIT).iterdir())
cache = []
for p in EVAL_IMAGES:
    r = model(str(p), conf=0.05, verbose=False)[0]  # conf를 낮게 받아 두고 나중에 거름
    # 예측 한 줄 = [x1, y1, x2, y2, conf, cls]
    # boxes.xyxy 모서리 픽셀 좌표, .conf 신뢰도, .cls 클래스 번호 (.cpu().numpy()로 배열화)
    pred = np.hstack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()[:, None], r.boxes.cls.cpu().numpy()[:, None]])
    cache.append((p, pred, gt_boxes(p, EVAL_SPLIT)))  # (이미지 경로, 예측, 정답)

# count: 임계값 thr에서 TP·FP·FN과 precision·recall·F1 계산
# cls를 주면 그 클래스만, 없으면 모든 클래스 합계 (같은 클래스끼리만 짝지음)
def count(thr, cls=None):
    tp = fp = fn = 0
    for _, pred, gt in cache:
        for c in (range(len(NAMES)) if cls is None else [cls]):  # 클래스별로 따로 매칭
            # P: thr 이상이고 클래스가 c인 예측, G: 클래스가 c인 정답
            P = pred[(pred[:, 4] >= thr) & (pred[:, 5] == c)]; G = gt[gt[:, 4] == c]
            # used: 이미 짝지어진 정답 번호, hit: 맞힌 수
            ious, used, hit = iou_mat(P[:, :4], G[:, :4]), set(), 0
            for i in np.argsort(-P[:, 4]):  # 신뢰도 높은 예측부터
                if len(G) == 0:  # 정답이 없으면 예측은 모두 FP
                    break
                # 아직 안 쓴 정답 중 IoU가 가장 큰 것 (쓴 것은 -1로 제외)
                j = int(np.argmax(np.where([k in used for k in range(len(G))], -1, ious[i])))
                if ious[i, j] >= 0.5 and j not in used:  # IoU 0.5 이상이면 맞힘(TP)
                    used.add(j); hit += 1
            tp += hit; fp += len(P) - hit; fn += len(G) - hit  # 남은 예측 = FP, 못 찾은 정답 = FN
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)  # max(…, 1): 0으로 나누기 방지
    return {'conf': thr, 'tp': tp, 'fp': fp, 'fn': fn, 'precision': round(prec, 3), 'recall': round(rec, 3),
            'f1': round(2 * prec * rec / max(prec + rec, 1e-9), 3)}  # F1 = 정밀도·재현율의 조화평균

thr_table = pd.DataFrame([count(t) for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]])  # 임계값 7개를 각각 계산해 표로
BEST_CONF = float(thr_table.loc[thr_table['f1'].idxmax(), 'conf'])  # idxmax: F1이 가장 큰 행
print('전체 F1이 가장 높은 임계값:', BEST_CONF)
thr_table

In [ ]:
FOCUS_CLASS = NAMES[0]          # ✏️ 가장 중요한 클래스 이름 (예: 'no_helmet') — 아래 표·실험·리포트의 기준
fc = NAMES.index(FOCUS_CLASS)  # 클래스 이름 → 번호
focus = pd.DataFrame([count(t, fc) for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]])  # 이 클래스만 따로 계산
print(f'"{FOCUS_CLASS}" 클래스만 따로 본 임계값별 성능')
focus

**생각해 보기** — 우리 문제에서 놓침(FN)과 헛경보(FP) 중 무엇이 더 비싼가요? 놓침이 위험하면 재현율이 목표를 넘는 가장 높은 임계값을, 헛경보가 비싸면 정밀도가 목표를 넘는 가장 낮은 임계값을 고릅니다. 고른 값을 아래 `SERVICE_CONF`에 적으세요.

In [ ]:
SERVICE_CONF = BEST_CONF        # ✏️ 위 표를 보고 우리 문제에 맞는 값으로 (데모·리포트에 쓰임)

### 6-3. 오류 분석 — 틀린 사진 직접 보기

In [ ]:
# 사진마다 '예측 수와 정답 수의 차이'로 오류 점수를 매겨 큰 순으로 정렬
errs = []
for p, pred, gt in cache:
    P = pred[pred[:, 4] >= SERVICE_CONF]  # 서비스 임계값 이상 예측만
    errs.append((abs(len(P) - len(gt)) + len(gt) - min(len(P), len(gt)), p, P, gt))
errs.sort(key=lambda e: -e[0])  # 오류 점수 내림차순

# 오류가 큰 6장: 정답은 초록, 예측은 빨강 (색은 BGR 순서)
imgs, titles = [], []
for e, p, P, gt in errs[:6]:
    im = cv2.imread(str(p))
    for x1, y1, x2, y2, c in gt:  # 정답 박스
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 3)
    for x1, y1, x2, y2, cf, c in P:  # 예측 박스 + 클래스·신뢰도 글자
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 2)
        cv2.putText(im, f'{NAMES[int(c)]} {cf:.2f}', (int(x1), max(12, int(y1) - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)
    imgs.append(im); titles.append(f'GT {len(gt)} (green) / pred {len(P)} (red) | {p.name[:20]}')
show_grid(imgs, titles)

**오류 분류표** ✏️ — 위 사진들을 보고 원인별로 몇 장인지 세어 적으세요. 7단계 개선 실험의 방향이 여기서 나옵니다.

| 원인 | 장 수 | 개선 아이디어 |
|---|---|---|
| 작은 물체를 놓침 | | `imgsz` 키우기 |
| 비슷한 물체와 헷갈림 | | 더 큰 모델, 헷갈리는 사례 데이터 추가 |
| 가려짐·겹침 | | 데이터 증강(augmentation — 뒤집기·밝기 변경 등으로 학습 이미지를 변형해 늘리기), 데이터 추가 |
| 라벨 자체가 틀림 | | 라벨 수정 |

---
## 7. 개선 실험 기록

한 번에 **하나만** 바꿉니다. 실험마다 이름, 바꾼 것, 이유를 적고, 결과는 표에 자동으로 쌓입니다.

In [ ]:
# 실험 하나 = 이름, 바꿀 학습 설정(change), 이유(why). 'model' = 시작 모델
EXPERIMENTS = [
    # ✏️ 이름, 바꿀 설정, 이유 — 오류 분류표를 근거로 1~3개
    {'name': 'imgsz_960', 'change': {'imgsz': 960}, 'why': '작은 물체 놓침이 많아서'},
    # {'name': 'model_s', 'change': {'model': 'yolo26s.pt'}, 'why': '비슷한 물체 혼동'},
    # {'name': 'more_epochs', 'change': {'epochs': 100}, 'why': '손실이 아직 내려가는 중'},
]
RUN_EXPERIMENTS = False         # ✏️ True면 위 실험을 차례로 학습 (실험마다 베이스라인만큼 걸림)

# evaluate: 가중치를 평가 세트로 검증 → mAP50, mAP50-95, 중요 클래스 AP50
def evaluate(weights):
    r = YOLO(weights).val(data=DATA_YAML, split=EVAL_SPLIT, verbose=False, plots=False)
    ap = dict(zip(r.box.ap_class_index.tolist(), r.box.ap50.tolist()))  # 클래스 번호 → AP50
    return {'map50': round(r.box.map50, 3), 'map50_95': round(r.box.map, 3), f'ap50_{FOCUS_CLASS}': round(ap.get(fc, 0), 3)}

# upsert: 같은 이름의 행이 이미 있으면 새 행으로 바꾸고, 없으면 뒤에 붙임
# (QUICK_RUN으로 먼저 돌린 기록이 본 학습 결과로 바뀌도록)
def upsert(log, row):
    names = [r['name'] for r in log]
    if row['name'] in names:
        log[names.index(row['name'])] = row  # 같은 자리에서 교체 (표 순서 유지)
    else:
        log.append(row)

# Drive의 experiments.csv를 읽어 이어서 기록
# to_dict('records'): 표를 '행 하나 = dict' 목록으로
log_path = SAVE_DIR / 'experiments.csv'
log = pd.read_csv(log_path).to_dict('records') if log_path.exists() else []
# 베이스라인 행은 방금 5단계에서 학습한 모델로 매번 다시 적음 (epochs 열 = 그때 학습 반복 수)
upsert(log, {'name': 'baseline', 'change': '-', 'why': '기준점', 'epochs': BASE_CFG['epochs'],
             'train_min': BASE_MIN, **evaluate(BEST)})
if RUN_EXPERIMENTS:
    for e in EXPERIMENTS:
        ch = dict(e['change']); mdl = ch.pop('model', BASE_MODEL)  # 'model'만 꺼내고 나머지는 학습 설정
        ep = ch.get('epochs', CFG['epochs'])  # 이번에 학습할 반복 수
        # 같은 이름·같은 epochs로 이미 기록된 실험은 다시 학습하지 않음 (QUICK_RUN 기록이면 다시 학습)
        if any(r['name'] == e['name'] and r.get('epochs') == ep for r in log):
            continue
        w, mins, _ = train(e['name'], model=mdl, **ch)  # 바꾼 설정으로 학습
        upsert(log, {'name': e['name'], 'change': json.dumps(e['change']), 'why': e['why'], 'epochs': ep,
                     'train_min': mins, **evaluate(w)})
exp = pd.DataFrame(log)
exp.to_csv(log_path, index=False)  # Drive에 저장 (열 이름은 영문)
exp

실험 기록은 Drive의 `experiments.csv`에 계속 쌓이므로, 런타임이 끊기거나 다음 시간에 이어서 해도 기록이 남습니다. 같은 이름으로 다시 학습하면 그 행은 새 결과로 바뀝니다. 예를 들어 `QUICK_RUN = True`로 먼저 돌린 뒤 `False`로 본 학습을 하면 `baseline` 행이 본 학습 결과로 바뀌고, `epochs` 열에서 몇 번 반복한 결과인지 확인할 수 있습니다.

가장 좋은 실험의 가중치(weights — 학습으로 얻은 모델 값, `.pt` 파일)를 다음 셀에서 고릅니다.

In [ ]:
best_row = exp.loc[exp['map50'].idxmax()]  # mAP50이 가장 높은 실험 행
FINAL_WEIGHTS = SAVE_DIR / f"{best_row['name']}_best.pt"  # 그 실험의 Drive 가중치 파일
print('최종 모델:', best_row['name'], '| mAP50', best_row['map50'], '|', FINAL_WEIGHTS)

### 7-2. 최종 모델로 다시 평가하기

6단계의 클래스별 지표와 임계값 표는 **베이스라인** 모델로 만든 것입니다. 최종 모델이 바뀌었다면 같은 계산을 최종 모델로 다시 해야 데모와 리포트의 숫자가 맞습니다. 아래 셀은 `per_class`, `thr_table`, `focus`를 최종 모델 기준으로 다시 만듭니다(최종 모델이 베이스라인이면 그대로 둡니다).

In [ ]:
FINAL_NAME = best_row['name']  # 최종 모델 실험 이름 (리포트 제목에 쓰임)
if FINAL_NAME != 'baseline':
    model = YOLO(FINAL_WEIGHTS)  # 6단계와 같은 이름(model)으로 최종 모델을 불러와 아래 계산이 이 모델을 쓰게 함
    # 클래스별 지표 다시 계산 (6-1과 같은 방법)
    res = model.val(data=DATA_YAML, split=EVAL_SPLIT, verbose=False, plots=True, project='runs_ws', name='eval_final', exist_ok=True)
    per_class = pd.DataFrame({
        'class': [NAMES[i] for i in res.box.ap_class_index],
        'precision': np.round(res.box.p, 3), 'recall': np.round(res.box.r, 3),
        'ap50': np.round(res.box.ap50, 3), 'ap50_95': np.round(res.box.ap, 3)})
    per_class.to_csv('per_class.csv', index=False); save_out('per_class.csv')
    # 평가 이미지 예측을 최종 모델로 다시 저장 (6-2의 cache를 새로 채움)
    cache = []
    for p in EVAL_IMAGES:
        r = model(str(p), conf=0.05, verbose=False)[0]
        pred = np.hstack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()[:, None], r.boxes.cls.cpu().numpy()[:, None]])
        cache.append((p, pred, gt_boxes(p, EVAL_SPLIT)))
    # 임계값 표 다시 계산 (count 함수는 새 cache를 씀)
    thr_table = pd.DataFrame([count(t) for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]])
    focus = pd.DataFrame([count(t, fc) for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]])
    print(f'최종 모델({FINAL_NAME}) 기준 {EVAL_SPLIT} 세트 | mAP50 {res.box.map50:.3f} | mAP50-95 {res.box.map:.3f}')
    print('전체 F1이 가장 높은 임계값:', float(thr_table.loc[thr_table['f1'].idxmax(), 'conf']))
    display(per_class); display(focus)
else:
    print('최종 모델이 베이스라인이므로 6단계 결과를 그대로 씁니다.')

최종 모델의 `focus` 표(중요 클래스의 임계값별 성능)를 보고 서비스 임계값을 다시 고릅니다. 6-2에서 정한 값이 그대로 좋으면 바꾸지 않아도 됩니다.

In [ ]:
SERVICE_CONF = SERVICE_CONF   # ✏️ 최종 모델 표를 보고 필요하면 숫자로 바꾸기 (예: 0.3) — 데모·리포트에 쓰임
print('서비스 임계값:', SERVICE_CONF)

---
## 8. 데모 — 새 이미지·영상에 적용

평가 세트에서 몇 장을 골라 최종 모델로 예측하고, 원하면 **직접 가져온 사진이나 영상**도 올려 봅니다.

In [ ]:
final = YOLO(FINAL_WEIGHTS)  # 최종 모델 불러오기
# 평가 이미지 6장 예측, plot()은 박스를 그린 BGR 이미지를 돌려줌
demo_imgs = [final(str(p), conf=SERVICE_CONF, verbose=False)[0].plot() for p in EVAL_IMAGES[:6]]
show_grid(demo_imgs, [p.name[:30] for p in EVAL_IMAGES[:6]])
for i, im in enumerate(demo_imgs[:3]):  # 처음 3장을 파일로 저장 + Drive 복사
    cv2.imwrite(f'demo_{i + 1}.jpg', im); save_out(f'demo_{i + 1}.jpg')

In [ ]:
UPLOAD_DEMO = False             # ✏️ True로 바꾸면 사진·영상 파일을 올려 예측
if UPLOAD_DEMO and IN_COLAB:
    up = files.upload()  # 파일 선택 창 → {파일 이름: 내용}
    for name in up:
        ext = Path(name).suffix.lower()  # 확장자 (.jpg, .mp4 등)
        # 한글·공백 없는 이름으로: upload_1.jpg, upload_2.mp4 …
        safe = f'upload_{len(list(Path(".").glob("upload_*"))) + 1}{ext}'
        os.replace(name, safe)  # 파일 이름 바꾸기
        # save=True: 박스를 그린 결과를 runs_ws/demo/에 저장 (영상도 가능)
        res_u = final.predict(safe, conf=SERVICE_CONF, save=True, project='runs_ws', name='demo', exist_ok=True, verbose=False)
        print('저장:', Path(res_u[0].save_dir))

---
## 9. 결과 정리 — 요약 리포트

프로젝트 카드, 데이터 요약, 실험 기록, 클래스별 성능, 성공 기준 달성 여부를 한 파일(`project_report.md`)로 자동 정리합니다. 결과 공유나 멘토링 때 이 파일과 `report.png`를 보여 주면 됩니다.

In [ ]:
focus_at = count(SERVICE_CONF, fc)  # 서비스 임계값에서 중요 클래스 성능
# 리포트 내용을 한 줄씩 목록으로 (마지막에 줄바꿈으로 이어 붙임)
lines = [
    f'# {PROJECT_NAME}', '',
    f'- Team: {TEAM}', f'- Problem: {PROBLEM}', f'- Success criterion: {SUCCESS}', '',
    '## Data',
    # 데이터 출처: DATA_SOURCE에 맞는 값을 사전에서 골라 표시
    f'- Source: {DATA_SOURCE} ({ {"kaggle": KAGGLE_SLUG, "url": ZIP_URL, "drive": DRIVE_ZIP, "ultralytics": ULTRA_YAML}[DATA_SOURCE] })',
    f'- Label format: {FMT} | classes ({len(NAMES)}): {", ".join(NAMES)}',
    f'- Split: {SPLIT_COUNTS} ({"original split" if KEPT_SPLIT else "random 7:2:1"})', '',
    '## Experiments', exp.to_markdown(index=False), '',  # to_markdown: 표 → 마크다운 표 (tabulate 필요)
    f'## Per-class performance ({EVAL_SPLIT}, final model: {FINAL_NAME})', per_class.to_markdown(index=False), '',
    f'## Service threshold: conf {SERVICE_CONF} (final model: {FINAL_NAME})',
    f'- {FOCUS_CLASS}: precision {focus_at["precision"]}, recall {focus_at["recall"]} (TP {focus_at["tp"]}, FP {focus_at["fp"]}, FN {focus_at["fn"]})', '',
    '## Notes (fill in)', '- What worked:', '- What did not work:', '- Next step:',
]
# UTF-8로 저장해야 한글이 깨지지 않음
Path('project_report.md').write_text('\n'.join(lines), encoding='utf-8'); save_out('project_report.md')

# report.png 왼쪽: 실험별 mAP50, 오른쪽: 임계값별 정밀도·재현율
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(exp['name'], exp['map50'], color='tab:blue'); axes[0].set_ylim(0, 1)
axes[0].set_title('mAP50 by experiment'); axes[0].tick_params(axis='x', rotation=20)
axes[1].plot(focus['conf'], focus['precision'], '-o', label='precision')
axes[1].plot(focus['conf'], focus['recall'], '-o', label='recall')
# 회색 점선 = 고른 서비스 임계값
axes[1].axvline(SERVICE_CONF, ls='--', c='gray'); axes[1].set_ylim(0, 1.05)
axes[1].set_title(f'{FOCUS_CLASS}: threshold trade-off'); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig('report.png', dpi=120); save_out('report.png'); plt.show()
print(open('project_report.md', encoding='utf-8').read())
print('\nDrive 저장 파일:', sorted(p.name for p in SAVE_DIR.iterdir()))  # Drive 결과 폴더의 파일 목록

---
## 마무리 체크리스트

- [ ] 프로젝트 카드의 성공 기준과 실제 결과를 비교해 `project_report.md`의 Notes를 채웠다
- [ ] 실험을 최소 1개 이상 해서 베이스라인과 비교했다
- [ ] 오류 분류표를 채우고, 개선 실험이 그 근거와 연결된다
- [ ] Drive에 `*_best.pt`, `experiments.csv`, `project_report.md`, `report.png`가 있다

---
## 용어 정리

| 용어 | 뜻 |
|---|---|
| YOLO 형식 | 이미지마다 같은 이름의 `.txt` 라벨. 한 줄이 물체 하나이고 `클래스 cx cy w h`(박스 중심·너비·높이를 이미지 크기로 나눈 0~1 값)입니다. |
| Pascal VOC | 이미지마다 `.xml` 라벨이 있고 박스를 `xmin ymin xmax ymax` 픽셀 좌표로 적는 형식입니다. 이 노트북이 YOLO 형식으로 자동 변환합니다. |
| COCO | `.json` 하나에 `images`, `annotations`, `categories`를 모두 담는 형식입니다. 박스는 `bbox = [x, y, w, h]` 픽셀(x·y는 왼쪽 위)이며 자동 변환됩니다. |
| 세그멘테이션 다각형 | 물체 윤곽을 꼭짓점 좌표로 적은 라벨입니다. YOLO txt인데 한 줄에 숫자가 5개보다 많으면 이 형식이며, 이 노트북은 감싸는 박스로 바꿉니다. |
| 정규화 좌표 | 픽셀 좌표를 이미지 너비·높이로 나눠 0~1로 만든 값입니다. 이미지 크기가 달라도 같은 방식으로 다룰 수 있습니다. |
| stem | 파일 이름에서 확장자를 뺀 부분입니다(`a.jpg` → `a`). 이미지와 라벨 파일을 짝지을 때 씁니다. |
| data.yaml | 데이터 위치(`path`), 분할 폴더(`train`, `val`, `test`), 클래스 번호와 이름(`names`)을 적은 설정 파일입니다. 학습·평가 때 `data=`로 넘깁니다. |
| 분할 (train / val / test) | train은 학습용, val은 학습 중 성능 확인과 최고 가중치 선택용, test는 마지막 평가용입니다. 원본에 분할이 없으면 7:2:1로 나눕니다. |
| SEED | 난수 시작값입니다. 같은 값이면 무작위 분할·샘플 선택이 매번 똑같이 나와 결과를 비교할 수 있습니다. |
| 데이터 누수 | 평가 데이터의 정보가 학습에 섞이는 것입니다. 같은 원본의 증강 사본이 train과 test에 나뉘어 들어가면 점수가 실제보다 높게 나옵니다. |
| EDA (데이터 살펴보기) | 탐색적 데이터 분석. 학습 전에 클래스별 박스 수, 박스 크기, 샘플 그림을 보고 데이터의 특징과 문제를 찾는 단계입니다. |
| 클래스 불균형 | 클래스마다 박스 수 차이가 큰 상태입니다. 박스가 적은 클래스는 성능이 낮게 나오기 쉬워 클래스별 지표를 따로 봐야 합니다. |
| 베이스라인 | 작은 모델과 기본 설정으로 처음 끝까지 학습한 기준 모델입니다. 개선 실험은 모두 이 점수와 비교합니다. |
| 사전학습 가중치 / best.pt | 가중치는 학습으로 얻은 모델의 값(`.pt` 파일)입니다. `yolo26n.pt`처럼 큰 데이터로 미리 학습된 가중치에서 시작하고, 학습 중 검증 성능이 가장 좋았던 것이 `best.pt`입니다. |
| epoch | 학습 데이터 전체를 한 번 모두 학습하는 단위입니다. `QUICK_RUN = True`면 5, `False`면 50입니다. |
| imgsz | 학습·예측 때 이미지를 맞추는 한 변 크기(픽셀)입니다. 키우면 작은 물체를 더 잘 찾지만 느려지고 메모리를 더 씁니다. |
| batch | 한 번에 GPU에 넣어 학습하는 이미지 장 수입니다. GPU 메모리가 부족하면(CUDA out of memory) 줄입니다. |
| fraction | 학습 이미지 중 실제로 쓸 비율입니다. 0.3이면 30%만 써서 빠르게 학습합니다. |
| conf / 신뢰도 임계값 | conf는 모델이 예측마다 매긴 0~1 확신 점수이고, 임계값은 이보다 낮은 예측을 버리는 기준입니다. 올리면 FP가 줄고 FN이 늘어납니다. |
| IoU | Intersection over Union. 예측과 정답이 겹친 넓이를 둘을 합친 넓이로 나눈 값입니다. 이 노트북은 0.5 이상이면 맞힌 것으로 봅니다. |
| TP / FP / FN | TP는 맞게 찾은 박스, FP는 없는 물체를 찾은 헛탐지(헛경보), FN은 있는데 놓친 물체입니다. |
| precision (정밀도) | 찾았다고 한 것 중 맞은 비율, TP ÷ (TP + FP)입니다. 헛경보가 적을수록 높습니다. |
| recall (재현율) | 실제 정답 중 찾아낸 비율, TP ÷ (TP + FN)입니다. 놓침이 적을수록 높습니다. |
| F1 | 정밀도와 재현율의 조화평균입니다. 둘 중 하나만 높아서는 커지지 않으므로 균형을 보는 지표입니다. |
| AP / mAP50 | AP는 한 클래스의 정밀도-재현율 곡선 아래 면적이고, mAP는 클래스별 AP의 평균입니다. mAP50은 IoU 0.5를 맞힘 기준으로 계산합니다. |
| mAP50-95 | IoU 기준을 0.5부터 0.95까지 0.05씩 바꿔 가며 구한 mAP의 평균입니다. 박스 위치를 더 엄격하게 봅니다. |
| 혼동 행렬 | confusion matrix. 정답 클래스(열)가 어떤 클래스로 예측됐는지(행) 센 표입니다. 어떤 클래스끼리 헷갈리는지(혼동) 보여 줍니다. |
| PR 곡선 | 임계값을 바꿔 가며 정밀도(세로)와 재현율(가로)을 이은 곡선입니다. 곡선 아래 면적이 AP입니다. |
| 서비스 임계값 (SERVICE_CONF) | 실제 사용할 때 적용할 신뢰도 임계값입니다. 놓침이 위험하면 낮게, 헛경보가 비싸면 높게 고릅니다. |
| 증강 | augmentation. 뒤집기·밝기 변경·잘라 붙이기 등으로 학습 이미지를 변형해 다양하게 만드는 것입니다. Ultralytics는 학습 중 자동으로 적용합니다. |
| kagglehub | 캐글 데이터셋을 `작성자/이름`(KAGGLE_SLUG)만으로 내려받는 파이썬 라이브러리입니다. |
| 보안 비밀 (Secrets) | Colab 왼쪽 열쇠 아이콘. API 토큰 같은 비밀 값을 코드 밖에 저장하고 `userdata.get()`으로 읽습니다. 노트북을 공유해도 값은 퍼지지 않습니다. |
| 런타임 | Colab이 빌려주는 실행 환경(가상 컴퓨터)입니다. 끊기면 로컬 파일(`raw_data`, `dataset`)은 사라지고 Drive에 저장한 것만 남습니다. |
| 행 교체 (upsert) | 같은 이름의 행이 있으면 새 값으로 바꾸고, 없으면 새로 추가하는 방식입니다. `experiments.csv`에서 같은 실험을 다시 돌렸을 때 옛 결과가 남지 않게 합니다. |
